In [ ]:
######################## 01 Add simple Structured Output
from google import genai
from pydantic import BaseModel, Field
from typing import Union, List, Optional
from dotenv import load_dotenv
import os

class NetworkingTopic(BaseModel):
    is_networking_related: bool = Field(description="Whether the input topic is related to computer networking.")
    category: Optional[str] = Field(description="The category of networking (e.g., Routing, Security, Wireless).")
    confidence_score: float = Field(description="Confidence score from 0.0 to 1.0.")  

client = genai.Client()

user_input = "Gravity" 

gatekeeper_prompt = f"Identify if the following topic is related to computer networking: '{user_input}'"

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents=gatekeeper_prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": NetworkingTopic.model_json_schema()
    }
)
#### Json
print("\nJSON Format")
print(response.text)
print(type(response.text))

### Pydantic Object
print("\nPydantic Object")
clasification_result = NetworkingTopic.model_validate_json(response.text)
print(clasification_result)
print(type(clasification_result))
print(clasification_result.is_networking_related)

### Python Dict
print(clasification_result.model_dump())





JSON Format
{"is_networking_related":false,"category":null,"confidence_score":1.0}
<class 'str'>

Pydantic Object
is_networking_related=False category=None confidence_score=1.0
<class '__main__.NetworkingTopic'>
False
{'is_networking_related': False, 'category': None, 'confidence_score': 1.0}


In [22]:
############# 

######################## 02 Add Gatekeeper
from google import genai
from pydantic import BaseModel, Field
from typing import Union, List, Optional
from dotenv import load_dotenv
import os

class NetworkingTopic(BaseModel):
    is_networking_related: bool = Field(description="Whether the input topic is related to computer networking.")
    category: Optional[str] = Field(description="The category of networking (e.g., Routing, Security, Wireless).")
    confidence_score: float = Field(description="Confidence score from 0.0 to 1.0.")  

class NetworkingExplanation(BaseModel):
    topic_name: str
    definition: str = Field(description="A clear and concise definition of the topic.")
    core_concepts: List[str] = Field(description="List of key principles or components.")
    config_example: str = Field(description="A CLI or code snippet example showing configuration.")
    best_practices: List[str] = Field(description="Recommendations for implementation.")
    
client = genai.Client()

user_input = "log" 

gatekeeper_prompt = f"Identify if the following topic is related to computer networking: '{user_input}'"

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents=gatekeeper_prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": NetworkingTopic.model_json_schema()
    }
)
#### Json
# print("\nJSON Format")
# print(response.text)
# print(type(response.text))

# ### Pydantic Object
# print("\nPydantic Object")
# classification_result = NetworkingTopic.model_validate_json(response.text)
# print(classification_result)
# print(type(classification_result))
# print(classification_result.is_networking_related)

# ### Python Dict
# print(classification_result.model_dump())
def generate_networking_doc(topic: str, client: genai.Client):
    """
    Triggered only if the Gatekeeper confirms the topic is networking related.
    """
    print(f"🚀 Triggering Expert Agent for: {topic}...")
    
    prompt = f"""
    You are a Senior Network Engineer. Generate a detailed technical explanation for: {topic}.
    Include a definition, core concepts, a practical configuration example (Cisco/Juniper/Linux), 
    and best practices.
    """
    
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_json_schema": NetworkingExplanation.model_json_schema(),
        },
    )
    
    explanation = NetworkingExplanation.model_validate_json(response.text)
    
    print("\n--- Technical Documentation ---")
    print(f"TOPIC: {explanation.topic_name}")
    print(f"DEFINITION: {explanation.definition}")
    print("\nCORE CONCEPTS:")
    for concept in explanation.core_concepts:
        print(f" - {concept}")
    print(f"\nCONFIG EXAMPLE:\n{explanation.config_example}")
    print("\nBEST PRACTICES:")
    for bp in explanation.best_practices:
        print(f" - {bp}")

def process_gatekeeper(topic: str, result: NetworkingTopic, client: genai.Client):
    """
    Decides whether to trigger the content generation agent.
    """
    print(f"🔍 GATEKEEPER ANALYSIS")
    print(f"Confidence: {result.confidence_score}")
    
    if result.is_networking_related and result.confidence_score > 0.7:
        print(f"✅ VALID: Identified as {result.category}")
        generate_networking_doc(topic, client)
    else:
        print("❌ REJECTED: This does not appear to be a networking topic.")

try:
    classification_result = NetworkingTopic.model_validate_json(response.text)
    print(classification_result)
    process_gatekeeper(user_input, classification_result, client)
except Exception as e:
    print(f"Error: {e}")



is_networking_related=True category='Network Management' confidence_score=0.85
🔍 GATEKEEPER ANALYSIS
Confidence: 0.85
✅ VALID: Identified as Network Management
🚀 Triggering Expert Agent for: log...

--- Technical Documentation ---
TOPIC: Log Management
DEFINITION: Log management encompasses the processes of generating, collecting, aggregating, transmitting, storing, analyzing, archiving, and ultimately disposing of log data. Logs provide a historical record of events occurring within a system or network, crucial for troubleshooting, security auditing, compliance reporting, and performance monitoring.

CORE CONCEPTS:
 - Log Generation: The creation of log events by applications, operating systems, network devices, and other systems.  Logs contain timestamps, event descriptions, severity levels, and other relevant data.
 - Log Collection & Aggregation:  The process of gathering logs from multiple sources and centralizing them in a single location (e.g., a syslog server or SIEM solution).

## Get Structured Output From Gemini 3 API


In [16]:
from google import genai
from pydantic import BaseModel, Field
from typing import Union, List, Optional
from dotenv import load_dotenv
import os

class NetworkingTopic(BaseModel):
    is_networking_related: bool = Field(description="Whether the input topic is related to computer networking.")
    category: Optional[str] = Field(description="The category of networking (e.g., Routing, Security, Wireless).")
    confidence_score: float = Field(description="Confidence score from 0.0 to 1.0.") 

class NetworkingExplanation(BaseModel):
    topic_name: str
    definition: str = Field(description="A clear and concise definition of the topic.")
    core_concepts: List[str] = Field(description="List of key principles or components.")
    config_example: str = Field(description="A CLI or code snippet example showing configuration.")
    best_practices: List[str] = Field(description="Recommendations for implementation.")
    
user_input = "spanning" 
client = genai.Client()

gatekeeper_prompt = f"Identify if the following topic is related to computer networking: '{user_input}'"

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents=gatekeeper_prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": NetworkingTopic.model_json_schema()
    })

def generate_networking_doc(topic: str, client: genai.Client):
    """
    Triggered only if the Gatekeeper confirms the topic is networking related.
    """
    print(f"🚀 Triggering Expert Agent for: {topic}...")
    
    prompt = f"""
    You are a Senior Network Engineer. Generate a detailed technical explanation for: {topic}.
    Include a definition, core concepts, a practical configuration example (Cisco/Juniper/Linux), 
    and best practices.
    """
    
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_json_schema": NetworkingExplanation.model_json_schema(),
        },
    )
    
    explanation = NetworkingExplanation.model_validate_json(response.text)
    
    print("\n--- Technical Documentation ---")
    print(f"TOPIC: {explanation.topic_name}")
    print(f"DEFINITION: {explanation.definition}")
    print("\nCORE CONCEPTS:")
    for concept in explanation.core_concepts:
        print(f" - {concept}")
    print(f"\nCONFIG EXAMPLE:\n{explanation.config_example}")
    print("\nBEST PRACTICES:")
    for bp in explanation.best_practices:
        print(f" - {bp}")



def process_gatekeeper(topic: str, result: NetworkingTopic, client: genai.Client):
    """
    Decides whether to trigger the content generation agent.
    """
    print(f"🔍 GATEKEEPER ANALYSIS")
    print(f"Confidence: {result.confidence_score}")
    
    if result.is_networking_related and result.confidence_score > 0.7:
        print(f"✅ VALID: Identified as {result.category}")
        generate_networking_doc(topic, client)
    else:
        print("❌ REJECTED: This does not appear to be a networking topic.")

try:
    classification_result = NetworkingTopic.model_validate_json(response.text)
    # print(classification_result)
    process_gatekeeper(user_input, classification_result, client)
except Exception as e:
    print(f"Error: {e}") 

    


🔍 GATEKEEPER ANALYSIS
Confidence: 0.95
✅ VALID: Identified as Switching
🚀 Triggering Expert Agent for: spanning...

--- Technical Documentation ---
TOPIC: Spanning Tree Protocol (STP)
DEFINITION: Spanning Tree Protocol (STP) is a Layer 2 network protocol that prevents loop formation in bridged or switched networks. Loops occur when there are multiple paths between network devices, leading to broadcast storms and MAC address table instability, ultimately causing network outages. STP algorithmically selects a single, loop-free path between any two points in the network, disabling redundant paths while providing redundancy in case of a link or device failure.

CORE CONCEPTS:
 - Bridge Protocol Data Units (BPDUs): Special data frames exchanged between switches to share information about the network topology. There are different types of BPDUs (Configuration, Topology Change Notification, Topology Change Acknowledgment).
 - Root Bridge: The central point of the STP topology. It is the switc

## Get Structured Output from Gemini 3 API
### Using Pydantic

In [15]:
from google import genai
from pydantic import BaseModel, Field
from typing import Union, List, Optional
from dotenv import load_dotenv
import os

class NetworkingTopic(BaseModel):
    is_networking_related: bool = Field(description="Whether the input topic is related to computer networking.")
    category: Optional[str] = Field(description="The category of networking (e.g., Routing, Security, Wireless).")
    confidence_score: float = Field(description="Confidence score from 0.0 to 1.0.")  

client = genai.Client()

user_input = "VLAN" 

gatekeeper_prompt = f"Identify if the following topic is related to computer networking: '{user_input}'"

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents=gatekeeper_prompt,
    config={
    "response_mime_type": "application/json",
    "response_json_schema": NetworkingTopic.model_json_schema()
    }
)

print("\nJSON Format")
print(response.text)
print(type(response.text))

### Pydantic Object
print("\nPydantic Object")
clasification_result = NetworkingTopic.model_validate_json(response.text)
print(clasification_result)
print(type(clasification_result))
print(clasification_result.is_networking_related)

### Python Dict
print(clasification_result.model_dump())



JSON Format
{"is_networking_related":true,"category":"Switching","confidence_score":1.0}
<class 'str'>

Pydantic Object
is_networking_related=True category='Switching' confidence_score=1.0
<class '__main__.NetworkingTopic'>
True
{'is_networking_related': True, 'category': 'Switching', 'confidence_score': 1.0}
